In [1]:
import random
from pathlib import Path
import yaml
import numpy as np
import torch
from huggingface_hub import hf_hub_download, snapshot_download

from PrithviWxC.dataloaders.merra2 import (
    Merra2Dataset,
    preproc,
    input_scalers,
    output_scalers,
    static_input_scalers,
)
from PrithviWxC.model import PrithviWxC
print('1')
# ==========================================
# 1. Environment & Device Setup
# ==========================================
torch.jit.enable_onednn_fusion(True)

if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = True
    torch.cuda.manual_seed(42)
else:
    device = torch.device("cpu")

random.seed(42)
torch.manual_seed(42)
np.random.seed(42)

print(f"Using device: {device}")

print('2')

# ==========================================
# 2. Variable Definitions & Task Config
# ==========================================
surface_vars = [
    "EFLUX", "GWETROOT", "HFLUX", "LAI", "LWGAB", "LWGEM", "LWTUP",
    "PS", "QV2M", "SLP", "SWGNT", "SWTNT", "T2M", "TQI", "TQL",
    "TQV", "TS", "U10M", "V10M", "Z0M"
]
static_surface_vars = ["FRACI", "FRLAND", "FROCEAN", "PHIS"]
vertical_vars = ["CLOUD", "H", "OMEGA", "PL", "QI", "QL", "QV", "T", "U", "V"]
levels = [34.0, 39.0, 41.0, 43.0, 44.0, 45.0, 48.0, 51.0, 53.0, 56.0, 63.0, 68.0, 71.0, 72.0]

padding = {"level": [0, 0], "lat": [0, -1], "lon": [0, 0]}
lead_times = [18]
input_times = [-6]
time_range = ("2020-01-01T00:00:00", "2020-01-02T05:59:59")
positional_encoding = "fourier"

variable_names = surface_vars + [
    f"{var}_level_{level}" for var in vertical_vars for level in levels
]

print('3')
# ==========================================
# 3. Data Directories & Downloads
# ==========================================
data_dir = Path("../data")
surf_dir = data_dir / "merra-2"
vert_dir = data_dir / "merra-2"
surf_clim_dir = data_dir / "climatology"
vert_clim_dir = data_dir / "climatology"

# Download required MERRA-2 and Climatology data from HuggingFace
snapshot_download(
    repo_id="ibm-nasa-geospatial/Prithvi-WxC-1.0-2300M",
    allow_patterns="merra-2/MERRA2_sfc_2020010[1].nc",
    local_dir=str(data_dir),
)
snapshot_download(
    repo_id="ibm-nasa-geospatial/Prithvi-WxC-1.0-2300M",
    allow_patterns="merra-2/MERRA_pres_2020010[1].nc",
    local_dir=str(data_dir),
)
snapshot_download(
    repo_id="ibm-nasa-geospatial/Prithvi-WxC-1.0-2300M",
    allow_patterns="climatology/climate_surface_doy00[1]*.nc",
    local_dir=str(data_dir),
)
snapshot_download(
    repo_id="ibm-nasa-geospatial/Prithvi-WxC-1.0-2300M",
    allow_patterns="climatology/climate_vertical_doy00[1]*.nc",
    local_dir=str(data_dir),
)
print('4')
# ==========================================
# 4. Dataset Instantiation
# ==========================================
dataset = Merra2Dataset(
    time_range=time_range,
    lead_times=lead_times,
    input_times=input_times,
    data_path_surface=surf_dir,
    data_path_vertical=vert_dir,
    climatology_path_surface=surf_clim_dir,
    climatology_path_vertical=vert_clim_dir,
    surface_vars=surface_vars,
    static_surface_vars=static_surface_vars,
    vertical_vars=vertical_vars,
    levels=levels,
    positional_encoding=positional_encoding,
)
assert len(dataset) > 0, "There doesn't seem to be any valid data."

print('5')
# ==========================================
# 5. Load Scalers
# ==========================================
surf_in_scal_path = data_dir / "climatology/musigma_surface.nc"
vert_in_scal_path = data_dir / "climatology/musigma_vertical.nc"
surf_out_scal_path = data_dir / "climatology/anomaly_variance_surface.nc"
vert_out_scal_path = data_dir / "climatology/anomaly_variance_vertical.nc"

hf_hub_download(repo_id="ibm-nasa-geospatial/Prithvi-WxC-1.0-2300M", filename=f"climatology/{surf_in_scal_path.name}", local_dir=str(data_dir))
hf_hub_download(repo_id="ibm-nasa-geospatial/Prithvi-WxC-1.0-2300M", filename=f"climatology/{vert_in_scal_path.name}", local_dir=str(data_dir))
hf_hub_download(repo_id="ibm-nasa-geospatial/Prithvi-WxC-1.0-2300M", filename=f"climatology/{surf_out_scal_path.name}", local_dir=str(data_dir))
hf_hub_download(repo_id="ibm-nasa-geospatial/Prithvi-WxC-1.0-2300M", filename=f"climatology/{vert_out_scal_path.name}", local_dir=str(data_dir))

in_mu, in_sig = input_scalers(surface_vars, vertical_vars, levels, surf_in_scal_path, vert_in_scal_path)
output_sig = output_scalers(surface_vars, vertical_vars, levels, surf_out_scal_path, vert_out_scal_path)
static_mu, static_sig = static_input_scalers(surf_in_scal_path, static_surface_vars)

print('6')
# ==========================================
# 6. Model Construction & Configuration
# ==========================================
hf_hub_download(repo_id="ibm-nasa-geospatial/Prithvi-WxC-1.0-2300M", filename="config.yaml", local_dir=str(data_dir))

with open(data_dir / "config.yaml", "r") as f:
    config = yaml.safe_load(f)

model = PrithviWxC(
    in_channels=config["params"]["in_channels"],
    input_size_time=config["params"]["input_size_time"],
    in_channels_static=config["params"]["in_channels_static"],
    input_scalers_mu=in_mu,
    input_scalers_sigma=in_sig,
    input_scalers_epsilon=config["params"]["input_scalers_epsilon"],
    static_input_scalers_mu=static_mu,
    static_input_scalers_sigma=static_sig,
    static_input_scalers_epsilon=config["params"]["static_input_scalers_epsilon"],
    output_scalers=output_sig**0.5,
    n_lats_px=config["params"]["n_lats_px"],
    n_lons_px=config["params"]["n_lons_px"],
    patch_size_px=config["params"]["patch_size_px"],
    mask_unit_size_px=config["params"]["mask_unit_size_px"],
    mask_ratio_inputs=0.0,
    mask_ratio_targets=0.0,
    embed_dim=config["params"]["embed_dim"],
    n_blocks_encoder=config["params"]["n_blocks_encoder"],
    n_blocks_decoder=config["params"]["n_blocks_decoder"],
    mlp_multiplier=config["params"]["mlp_multiplier"],
    n_heads=config["params"]["n_heads"],
    dropout=config["params"]["dropout"],
    drop_path=config["params"]["drop_path"],
    parameter_dropout=config["params"]["parameter_dropout"],
    residual="climate",
    masking_mode="global",
    encoder_shifting=True,
    decoder_shifting=True,
    positional_encoding=positional_encoding,
    checkpoint_encoder=[],
    checkpoint_decoder=[],
)
print('7')
# ==========================================
# 7. Load Weights
# ==========================================
weights_path = data_dir / "weights/prithvi.wxc.2300m.v1.pt"

state_dict = torch.load(weights_path, map_location="cpu", weights_only=False)
if "model_state" in state_dict:
    state_dict = state_dict["model_state"]

model.load_state_dict(state_dict, strict=True)
model = model.to(device)
print('8')
# ==========================================
# 8. Batch Preprocessing & Inference
# ==========================================
data = next(iter(dataset))
batch = preproc([data], padding)

for k, v in batch.items():
    if isinstance(v, torch.Tensor):
        batch[k] = v.to(device)

with torch.no_grad():
    model.eval()
    out = model(batch)

print(f"Inference complete! Output tensor shape: {out.shape}")

1
Using device: cuda
2
3


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

4
5
6
7
8


/home/gpuuser7/gpuuser7_a/miniconda3/envs/alok/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


Inference complete! Output tensor shape: torch.Size([1, 160, 360, 576])


# Quantize weights selectively

In [2]:
import torch

input_weights_path = "../data/weights/prithvi_inference_only.pt"
mixed_bf16_weights_path = "../data/weights/prithvi_mixed_bf16.pt"

state_dict = torch.load(input_weights_path, map_location='cpu', weights_only=True)
mixed_state_dict = {}

# Strict safeguard list: keep these layer types in FP32
FP32_KEYWORDS = ["scaler", "norm", "embedding", "bias", "mask_token"]

for name, tensor in state_dict.items():
    if tensor.is_floating_point() and tensor.numel() > 0:
        # Check if the parameter belongs to a sensitive layer
        needs_fp32 = any(key in name.lower() for key in FP32_KEYWORDS)
        
        if not needs_fp32:
            mixed_state_dict[name] = tensor.to(torch.bfloat16)
        else:
            mixed_state_dict[name] = tensor
    else:
        mixed_state_dict[name] = tensor

torch.save(mixed_state_dict, mixed_bf16_weights_path)
print("Successfully generated optimal Mixed BF16 weights file.")

Successfully generated optimal Mixed BF16 weights file.


# Run inference with autocast

In [3]:
import torch
from torch.amp import autocast
from PrithviWxC.dataloaders.merra2 import preproc

# Load model weights
state_dict = torch.load(mixed_bf16_weights_path, map_location='cpu', weights_only=True)
model.load_state_dict(state_dict, strict=True, assign=True)
model = model.to(device)

# Prepare batch inputs in FP32
data = next(iter(dataset))
batch = preproc([data], padding)
for k, v in batch.items():
    if isinstance(v, torch.Tensor):
        batch[k] = v.to(device)

# Perform inference
with torch.no_grad():
    model.eval()
    # autocast handles intermediate activations dynamically
    with autocast(device_type='cuda', dtype=torch.bfloat16):
        out_bf16 = model(batch)
    
    # Ensure final output is in FP32 before returning to CPU/NumPy
    out_bf16 = out_bf16.to(torch.float32).cpu().numpy()

print("Inference complete!")

Inference complete!


In [4]:
import numpy as np
import pandas as pd
import torch

# ==============================================================================
# OPTIONAL: Run FP32 Baseline (Uncomment if out_fp32 is no longer in memory)
# ==============================================================================
fp32_weights_path = "../data/weights/prithvi_inference_only.pt"
model.load_state_dict(torch.load(fp32_weights_path, map_location='cpu', weights_only=True), strict=True, assign=True)
model = model.to(device)
with torch.no_grad():
    model.eval()
    out_fp32 = model(batch).cpu().numpy()

# Re-load Mixed BF16 weights and run to ensure out_bf16 is also fresh
model.load_state_dict(torch.load(mixed_bf16_weights_path, map_location='cpu', weights_only=True), strict=True, assign=True)
model = model.to(device)
with torch.no_grad():
    model.eval()
    with autocast(device_type='cuda', dtype=torch.bfloat16):
        out_bf16 = model(batch).to(torch.float32).cpu().numpy()
# ==============================================================================

print("--- Calculating Precision Loss Metrics (FP32 vs Mixed BF16) ---\n")
results = []

# Iterate through every atmospheric variable
for i, var_name in enumerate(variable_names):
    var_fp32 = out_fp32[0, i]
    var_bf16 = out_bf16[0, i]
    
    # Calculate basic regression metrics
    mae = np.mean(np.abs(var_fp32 - var_bf16))
    rmse = np.sqrt(np.mean((var_fp32 - var_bf16)**2))
    max_diff = np.max(np.abs(var_fp32 - var_bf16))
    
    # Calculate standard deviation for robust relative error normalization
    std_dev = np.std(var_fp32)
    
    # Prevent division by zero for uniform fields using an epsilon (1e-8)
    rel_std_err = (rmse / (std_dev + 1e-8) * 100) if std_dev > 1e-7 else 0.0
    
    results.append({
        "Variable": var_name,
        "MAE": mae,
        "RMSE": rmse,
        "Max Diff": max_diff,
        "Std Dev": std_dev,
        "Rel Error (% Std)": rel_std_err
    })

# ==========================================
# 1. Global Tensor Metrics
# ==========================================
global_mae = np.mean(np.abs(out_fp32 - out_bf16))
global_rmse = np.sqrt(np.mean((out_fp32 - out_bf16)**2))
global_max_diff = np.max(np.abs(out_fp32 - out_bf16))

print("OVERALL MODEL DIFFERENCE (FP32 vs Mixed BF16)")
print("=" * 45)
print(f"Global MAE      : {global_mae:.6f}")
print(f"Global RMSE     : {global_rmse:.6f}")
print(f"Global Max Diff : {global_max_diff:.6f}\n")

# ==========================================
# 2. Variable-Wise (Layer-Wise) Accuracy
# ==========================================
df_results = pd.DataFrame(results)
print("TOP 15 VARIABLES WITH HIGHEST RELATIVE DEVIATION")
print("=" * 60)
print(df_results.sort_values(by="Rel Error (% Std)", ascending=False).head(15).to_string(index=False))
print("\n")

# ==========================================
# 3. Final Overall Model Accuracy
# ==========================================
# Convert percentages back to fractions for the accuracy calculation
average_relative_error = np.mean(df_results["Rel Error (% Std)"]) / 100.0

# Calculate relative accuracy (FP32 baseline = 1.0)
overall_relative_accuracy = 1.0 - average_relative_error
overall_accuracy_percentage = overall_relative_accuracy * 100

print("=" * 60)
print(f"{'MIXED BFLOAT16 RELATIVE ACCURACY':^60}")
print("=" * 60)
print(f"FP32 Baseline Accuracy : 1.000000 (100.00%)")
print(f"Average Relative Error : {average_relative_error:.6f} ({average_relative_error * 100:.4f}%)")
print("-" * 60)
print(f"Overall Model Accuracy : {overall_relative_accuracy:.6f} ({overall_accuracy_percentage:.4f}%)")
print("=" * 60)

--- Calculating Precision Loss Metrics (FP32 vs Mixed BF16) ---

OVERALL MODEL DIFFERENCE (FP32 vs Mixed BF16)
Global MAE      : 1.700145
Global RMSE     : 7.448474
Global Max Diff : 270.057190

TOP 15 VARIABLES WITH HIGHEST RELATIVE DEVIATION
        Variable          MAE         RMSE  Max Diff  Std Dev  Rel Error (% Std)
CLOUD_level_34.0 2.080235e-04 3.031771e-04  0.012671 0.000703          43.155426
   QI_level_48.0 6.984834e-07 1.454376e-06  0.000027 0.000006          25.139471
CLOUD_level_48.0 1.926794e-02 4.437805e-02  0.649891 0.179391          24.738155
CLOUD_level_45.0 2.664712e-02 5.639446e-02  0.674424 0.232965          24.207296
   PL_level_34.0 3.977175e-05 1.393549e-04  0.000488 0.000600          23.212397
   QI_level_51.0 4.600462e-07 8.686052e-07  0.000019 0.000004          23.182219
CLOUD_level_51.0 1.452125e-02 3.440787e-02  0.499326 0.151676          22.685137
OMEGA_level_39.0 2.310652e-03 3.696815e-03  0.067755 0.016557          22.328337
   QL_level_56.0 1.630031e-

In [5]:
import os

# Define the paths to your four weight files
original_weights_path = "../data/weights/prithvi.wxc.2300m.v1.pt"
fp32_inference_path = "../data/weights/prithvi_inference_only.pt"
mixed_fp16_path = "../data/weights/prithvi_mixed_precision.pt"
mixed_bf16_path = "../data/weights/prithvi_mixed_bf16.pt"

# Create a dictionary for easy iteration
model_files = {
    "Original Checkpoint (FP32 + Metadata)": original_weights_path,
    "Inference Only (Pure FP32)": fp32_inference_path,
    "Mixed Precision (FP16 + FP32 Scalers)": mixed_fp16_path,
    "Mixed Precision (BF16 + FP32 Scalers)": mixed_bf16_path
}

print("=" * 65)
print(f"{'MODEL WEIGHT FILE SIZES':^65}")
print("=" * 65)

for name, path in model_files.items():
    if os.path.exists(path):
        # Get size in bytes and convert to Gigabytes (GB)
        size_bytes = os.path.getsize(path)
        size_gb = size_bytes / (1024 ** 3)
        print(f"{name:<40} : {size_gb:>5.2f} GB")
    else:
        print(f"{name:<40} : [FILE NOT FOUND]")

print("=" * 65)

                     MODEL WEIGHT FILE SIZES                     
Original Checkpoint (FP32 + Metadata)    : 26.49 GB
Inference Only (Pure FP32)               :  8.86 GB
Mixed Precision (FP16 + FP32 Scalers)    :  4.46 GB
Mixed Precision (BF16 + FP32 Scalers)    :  4.46 GB


# Core meteorological functions

In [6]:
import numpy as np
import pandas as pd
import torch

def calculate_lat_weighted_rmse(pred, target, lat_array):
    """
    Calculates the Latitude-Weighted Root Mean Squared Error.
    Weights are based on the cosine of the latitude to account for grid cell area.
    """
    # Convert latitudes to radians and calculate cosine weights
    weights = np.cos(np.deg2rad(lat_array))
    
    # Normalize weights so their mean is 1 (maintains the original error magnitude)
    weights = weights / np.mean(weights)
    
    # Reshape to (n_lats, 1) so it broadcasts across the longitude dimension
    weights_2d = weights[:, np.newaxis]
    
    # Calculate weighted MSE
    sq_error = (pred - target) ** 2
    weighted_mse = np.mean(sq_error * weights_2d)
    
    return np.sqrt(weighted_mse)

def calculate_acc(pred, target, lat_array, climatology=None):
    """
    Calculates the Latitude-Weighted Anomaly Correlation Coefficient (ACC).
    If climatology is not provided, it falls back to spatial correlation 
    against the field's spatial mean.
    """
    if climatology is None:
        clim_pred = np.mean(pred)
        clim_target = np.mean(target)
    else:
        clim_pred = climatology
        clim_target = climatology

    # Compute anomalies
    pred_anom = pred - clim_pred
    target_anom = target - clim_target
    
    # Latitude weights
    weights = np.cos(np.deg2rad(lat_array))
    weights_2d = weights[:, np.newaxis]
    
    # Weighted Covariance and Variances
    cov = np.sum(weights_2d * pred_anom * target_anom)
    var_pred = np.sum(weights_2d * (pred_anom ** 2))
    var_target = np.sum(weights_2d * (target_anom ** 2))
    
    # Prevent division by zero on uniform fields
    if var_pred == 0 or var_target == 0:
        return 0.0
        
    acc = cov / np.sqrt(var_pred * var_target)
    return acc

# Evaluation script

In [7]:
print("--- Calculating Rigorous Meteorological Metrics ---\n")

# Assuming out_fp32 and out_bf16 (or out_fp16) are already generated
# Extract the latitude array corresponding to the 360-pixel height
n_lats = out_fp32.shape[-2]
lat_array = np.linspace(-90, 90, n_lats)

results = []

for i, var_name in enumerate(variable_names):
    # Extract the 2D spatial grid for the specific variable
    # Shape: (360, 576)
    var_benchmark = out_fp32[0, i]  # Swap with ground_truth[0, i] for true accuracy
    var_quantized = out_bf16[0, i]  # Swap with out_fp16 if testing the FP16 model
    
    # Calculate Lat-Weighted RMSE
    lw_rmse = calculate_lat_weighted_rmse(var_quantized, var_benchmark, lat_array)
    
    # Calculate ACC (without explicit temporal climatology for this hardware benchmark)
    acc = calculate_acc(var_quantized, var_benchmark, lat_array)
    
    # Calculate Standard Deviation for context
    std_dev = np.std(var_benchmark)
    
    results.append({
        "Variable": var_name,
        "Lat-Weighted RMSE": lw_rmse,
        "ACC": acc,
        "Std Dev": std_dev
    })

# ==========================================
# Compute Global Aggregates
# ==========================================
df_results = pd.DataFrame(results)

# Global Lat-Weighted RMSE (Root of the mean of weighted variances)
global_lw_rmse = np.sqrt(np.mean(df_results["Lat-Weighted RMSE"]**2))

# Global ACC (Mean across all physical variables)
global_acc = np.mean(df_results["ACC"])

print("============================================================")
print(f"{'GLOBAL METEOROLOGICAL ACCURACY (FP32 vs BF16)':^60}")
print("============================================================")
print(f"Global Lat-Weighted RMSE : {global_lw_rmse:.6f}")
print(f"Global ACC Score         : {global_acc:.6f} (1.0 is perfect correlation)")
print("============================================================\n")

print("TOP 15 VARIABLES WITH HIGHEST LAT-WEIGHTED RMSE")
print("-" * 60)
print(df_results.sort_values(by="Lat-Weighted RMSE", ascending=False).head(15).to_string(index=False))
print("\n")

print("TOP 15 VARIABLES WITH LOWEST ACC SCORE (Worst Pattern Match)")
print("-" * 60)
print(df_results.sort_values(by="ACC", ascending=True).head(15).to_string(index=False))

--- Calculating Rigorous Meteorological Metrics ---

       GLOBAL METEOROLOGICAL ACCURACY (FP32 vs BF16)        
Global Lat-Weighted RMSE : 6.877727
Global ACC Score         : 0.991683 (1.0 is perfect correlation)

TOP 15 VARIABLES WITH HIGHEST LAT-WEIGHTED RMSE
------------------------------------------------------------
     Variable  Lat-Weighted RMSE      ACC     Std Dev
          SLP          35.704301 0.999527 1268.997314
           PS          33.346052 0.999989 9434.502930
PL_level_72.0          33.112753 0.999989 9363.717773
PL_level_71.0          32.464197 0.999989 9191.131836
PL_level_68.0          30.331412 0.999989 8582.232422
PL_level_63.0          26.761972 0.999989 7572.484863
PL_level_56.0          19.993891 0.999989 5645.961914
PL_level_53.0          15.725358 0.999989 4426.383301
        SWGNT          12.599970 0.998727  218.975296
PL_level_51.0          12.288448 0.999989 3473.226074
        SWTNT          11.534704 0.999407  295.430847
PL_level_48.0           7.4